In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

In [ ]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    from pathlib import Path
    from langchain.document_loaders import PyPDFLoader

    documents = []

    pdf_files = Path(pdf_directory).glob('**/*.pdf')

    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        pages = loader.load()

        for page in pages:
            page.metadata['source_file'] = pdf_file.name
            page.metadata['file_type'] = 'pdf'

        documents.extend(pages)

    return documents

In [ ]:

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    from langchain.text_splitter import RecursiveCharacterTextSplitter

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs

In [ ]:
# CELL 13
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        from sentence_transformers import SentenceTransformer

        print(f"Loading model: {self.model_name}")

        self.model = SentenceTransformer(self.model_name)

        print("Model loaded successfully!")
        print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        """
        if self.model is None:
            raise ValueError("Model is not loaded!")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        return np.array(embeddings)

In [ ]:

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents",
                 persist_directory: str = "../data/vector_store"):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        import os
        import chromadb

        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "PDF document embeddings"
            }
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        """
        import uuid

        if len(documents) != len(embeddings):
            raise ValueError("Documents and embeddings count mismatch!")

        ids = []
        metadatas = []
        docs = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = uuid.uuid4().hex[:8]

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            ids.append(doc_id)
            metadatas.append(metadata)
            docs.append(doc.page_content)
            embedding_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            embeddings=embedding_list,
            metadatas=metadatas,
            documents=docs
        )

        print(f"Added {len(documents)} documents to vector store")

In [ ]:

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore,
                 embedding_manager: EmbeddingManager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str,
                 top_k: int = 5,
                 score_threshold: float = 0.0):

        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        for i in range(len(results["ids"][0])):

            distance = results["distances"][0][i]
            similarity_score = 1 - distance

            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "id": results["ids"][0][i],
                    "content": results["documents"][0][i],
                    "metadata": results["metadatas"][0][i],
                    "similarity_score": similarity_score,
                    "distance": distance,
                    "rank": i + 1
                })

        return retrieved_docs

In [ ]:

def rag_simple(query, retriever, llm, top_k=3):

    retrieved_docs = retriever.retrieve(
        query=query,
        top_k=top_k
    )

    context = "\n\n".join([
        doc["content"] for doc in retrieved_docs
    ])

    prompt = f"""
    Use the following context to answer the question.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content

In [ ]:

def rag_advanced(query, retriever, llm,
                 top_k=5,
                 min_score=0.2,
                 return_context=False):

    retrieved_docs = retriever.retrieve(
        query=query,
        top_k=top_k,
        score_threshold=min_score
    )

    context = "\n\n".join([
        doc["content"] for doc in retrieved_docs
    ])

    sources = []

    for doc in retrieved_docs:
        sources.append({
            "source_file": doc["metadata"].get("source_file", "Unknown"),
            "page": doc["metadata"].get("page", "N/A"),
            "similarity_score": doc["similarity_score"],
            "preview": doc["content"][:150]
        })

    confidence = max(
        [doc["similarity_score"] for doc in retrieved_docs],
        default=0
    )

    prompt = f"""
    Use the following context to answer the question.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    result = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        result["context"] = context

    return result

In [ ]:

from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self,
              question: str,
              top_k: int = 5,
              min_score: float = 0.2,
              stream: bool = False,
              summarize: bool = False) -> Dict[str, Any]:

        retrieved_docs = self.retriever.retrieve(
            query=question,
            top_k=top_k,
            score_threshold=min_score
        )

        context = "\n\n".join([
            doc["content"] for doc in retrieved_docs
        ])

        sources = []

        for doc in retrieved_docs:
            sources.append({
                "source_file": doc["metadata"].get("source_file", "Unknown"),
                "page": doc["metadata"].get("page", "N/A"),
                "score": doc["similarity_score"]
            })

        prompt = f"""
        Use the context below to answer the question.

        Context:
        {context}

        Question:
        {question}

        Answer:
        """

        if stream:
            for i in range(0, len(prompt), 100):
                print(prompt[i:i+100])
                time.sleep(0.1)

        response = self.llm.invoke(prompt)
        answer = response.content

        citations = "\n\nSources:\n"
        for src in sources:
            citations += (
                f"- {src['source_file']} "
                f"(Page {src['page']})\n"
            )

        answer_with_citations = answer + citations

        summary = None

        if summarize:
            summary_prompt = f"""
            Summarize the following answer in 2 sentences:

            {answer}
            """

            summary_response = self.llm.invoke(summary_prompt)
            summary = summary_response.content

        history_item = {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary
        }

        self.history.append(history_item)

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }